## Week 36
Familiarize yourself with the dataset card, download the dataset and explore its
columns. Summarize data statistics (size, word count, etc.) for training and
validation data in the languages Arabic (ar), Korean (ko) and Telugu (te).

For each of the languages Arabic, Korean and Telugu, report the 5 most
common words in the questions from the training set and their count, as well
as their English translation. What kind of words are they?

Implement a rule-based classifier that predicts whether a question is an-
swerable or impossible, only using the document (context) and question.

You
may use machine translation as a component. Use the answerable field to
evaluate it on the validation set. What is the performance of your classifier for
each of the languages Arabic, Korean and Telugu?

In [ ]:
from datasets import load_dataset
dataset = load_dataset("coastalcph/tydi_xor_rc")
train_set = dataset["train"]
validation_set = dataset["validation"]

In [7]:
import pandas as pd
import string
import numpy as np

In [8]:
validation_set = pd.read_parquet("data/validation.parquet")
train_set = pd.read_parquet("data/train.parquet")

In [9]:
strip_punctuation = lambda line: [word.strip(string.punctuation+"؟")for word in line.split(" ")]
count_words = lambda line: len(line)

In [10]:
#!pip install torch

In [11]:
from transformers import pipeline
import torch

In [12]:
import torch
print("Torch:", torch.__version__)        # should show +cu126
print("CUDA runtime:", torch.version.cuda) # '12.6'
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.8.0+cu126
CUDA runtime: 12.6
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [42]:
from datasets import Dataset
from transformers import pipeline
import pandas as pd
import torch
from itertools import chain


In [43]:
LANG_CODE = {"ko": "kor_Hang", "ar": "arb_Arab", "te": "tel_Telu"}
device = 0 if torch.cuda.is_available() else -1

translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    tokenizer="facebook/nllb-200-distilled-600M",
    device=device
)



Device set to use cuda:0


In [44]:
def hf_translate_questions(lang_df, l, batch_size: int = 32):
    work_df = lang_df[["question"]].copy()
    work_df["question"] = work_df["question"].fillna("").astype(str)

    ds = Dataset.from_pandas(work_df.reset_index(drop=True), preserve_index=False)

    def _translate_batch(batch):
        outs = translator(
            batch["question"],
            src_lang=LANG_CODE[l],
            tgt_lang="eng_Latn",
            max_length=256,
            truncation=True,
        )
        return {"question_translated": [o["translation_text"] for o in outs]}

    ds_out = ds.map(_translate_batch, batched=True, batch_size=batch_size)

    translated_df = ds_out.to_pandas()
    return translated_df[["question_translated"]]



In [50]:
def safe_top_tokens(lang_df, l, topn: int = 5):
    col = "question_stripped"
    if lang_df[col].dtype == object and lang_df[col].map(lambda x: isinstance(x, list)).any():
        tokens = list(chain.from_iterable(x or [] for x in lang_df[col].tolist()))
    else:
        tokens = list(chain.from_iterable((str(x or "")).split() for x in lang_df[col].tolist()))
    tokens = [t.strip() for t in tokens if isinstance(t, str) and t.strip()]
    counts = pd.Series(tokens).value_counts().head(topn)

    def translate_top_words(counts_series, lcode):
        tx = [
            translator(w, src_lang=LANG_CODE[lcode], tgt_lang="eng_Latn", max_length=64)[0]["translation_text"]
            for w in counts_series.index
        ]
        out = counts_series.reset_index()
        out.columns = ["token", "count"]
        out["translation"] = tx
        return out

    return translate_top_words(counts, l)


In [51]:

def statistics(df, col=["question", "context"], translate = False):
    langs = ["ko", "ar", "te"]
    df = df[df["lang"].isin(langs)].copy()

    for c in col:
        df[f"{c}_stripped"] = df[c].apply(strip_punctuation)
        df[f"{c}_wordcount"] = df[f"{c}_stripped"].apply(count_words)

    print(
        df.groupby(["lang", "answerable"]).agg(
            question_wordcount_mean=("question_wordcount", "mean"),
            context_wordcount_mean=("context_wordcount", "mean"),
            question_wordcount_sum=("question_wordcount", "sum"),
            context_wordcount_sum=("context_wordcount", "sum"),
            question_wordcount_max=("question_wordcount", "max"),
            question_wordcount_min=("question_wordcount", "min"),
            count=("question_wordcount", "count")
        ).round(0)
    )

    if translate:
        df["question_translated"] = None

        for l in langs:
            mask = df["lang"] == l
            lang_df = df.loc[mask].copy()

            print(f"\nTop tokens for {l} -> en:")
            top_counts = safe_top_tokens(lang_df, l)
            print(top_counts)

            if translate:
                translated_df = hf_translate_questions(lang_df, l, batch_size=32)
                df.loc[mask, "question_translated"] = translated_df["question_translated"].values
                torch.cuda.empty_cache()

    return df

In [23]:
translate = False

In [33]:
val_for_stat = statistics(validation_set, translate=translate)


                 question_wordcount_mean  context_wordcount_mean  \
lang answerable                                                    
ar   False                           8.0                   112.0   
     True                            7.0                   103.0   
ko   False                           5.0                   108.0   
     True                            5.0                    95.0   
te   False                           6.0                   112.0   
     True                            6.0                   105.0   

                 question_wordcount_sum  context_wordcount_sum  \
lang answerable                                                  
ar   False                          406                   5813   
     True                          2436                  37285   
ko   False                           95                   2051   
     True                          1636                  32124   
te   False                          575                  10

In [31]:
train_for_stat = statistics(train_set, translate=translate)

                 question_wordcount_mean  context_wordcount_mean  \
lang answerable                                                    
ar   False                           8.0                   124.0   
     True                            7.0                   102.0   
ko   False                           5.0                   104.0   
     True                            5.0                    97.0   
te   False                           6.0                   118.0   
     True                            6.0                    87.0   

                 question_wordcount_sum  context_wordcount_sum  \
lang answerable                                                  
ar   False                         1948                  31580   
     True                         15509                 234189   
ko   False                          325                   6574   
     True                         11524                 228653   
te   False                          277                   5

In [ ]:
if translate:
    val_for_stat.to_csv("data/validation_translated.csv")
    train_for_stat.to_csv("data/train_translated.csv")

  I wanted to strip punctuation, but certain languages that we do not include in the analysis have special punctuation that has to be included in thw stripping pool

In [53]:
val_tr = pd.read_csv("data/validation_translated.csv")
train_tr = pd.read_csv("data/train_translated.csv")

1) set of words for question and context
2) delete common words(give no value)
3) look if words from question are in the stripped context
4)

In [117]:
from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
tokenizer = AutoTokenizer.from_pretrained("PrimeQA/mt5-base-tydi-question-generator")

sequence = "Using a Transformer network is simple"
tokenize_f =lambda x:  tokenizer.tokenize(x)


tokenizer_config.json:   0%|          | 0.00/408 [00:00<?, ?B/s]

C:\Users\dnedi\PycharmProjects\nlp-course-project\newvenv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dnedi\.cache\huggingface\hub\models--PrimeQA--mt5-base-tydi-question-generator. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not 

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/103 [00:00<?, ?B/s]

In [118]:
tokenize_f(sequence)

['▁Us', 'ing', '▁', 'a', '▁Transformer', '▁network', '▁is', '▁simple']

In [119]:
train_tr["question_translated_tokenized"] = train_tr["question_translated"].apply(tokenize_f)
train_tr["context_tokenized"] = train_tr["context"].apply(tokenize_f)

In [121]:
# train_tr["question_translated_tokenized"]
# train_tr["context_tokenized"]

0       [▁The, ▁conflict, ▁, between, ▁France, ▁and, ▁...
1       [▁X, -, rays, ▁make, ▁up, ▁X, -, radiation, ,,...
2       [▁In, ▁2022, ,, ▁Beijing, ▁will, ▁, become, ▁t...
3       [▁The, ▁, British, ▁, Broadcast, ing, ▁Corpora...
4       [▁, Palestine, ▁(, ▁, '),, ▁official, ly, ▁the...
                              ...                        
6330    [▁In, ▁, February, ▁2012, ,, ▁Somali, ▁governm...
6331    [▁The, ▁first, ▁earth, ▁, tracks, ▁were, ▁, cr...
6332    [▁Abdel, ▁Fat, tah, ▁Sa, e, ed, ▁Hu, ssein, ▁K...
6333    [▁, Munich, ▁(, ;, ▁, ;, ▁, ), ▁is, ▁the, ▁cap...
6334    [▁Ana, ba, pt, ism, ▁(, from, ▁Neo, -, Latin, ...
Name: context_tokenized, Length: 6335, dtype: object

In [54]:
train_tr[["context", "answerable", "question_translated"]][train_tr["answerable"]== False]

,context,answerable,question_translated
17,Moonlight consists of mostly sunlight (with li...,False,Does the moon glow by itself?
36,With the end of the Macedonian Wars – which ra...,False,Was Rome victorious in the Phoenician War?
109,"When on land, an American alligator moves eith...",False,Can crocodiles live on both land and water?
115,Many species produce metabolites that are majo...,False,Can mushrooms heal wounds?
119,The University of Oxford has no known foundati...,False,Who is the founder of Oxford University?
...,...,...,...
6321,Vaccines against anthrax for use in livestock ...,False,In what country was the antidote to tuberculos...
6322,are broken Other Names of the Qur'an: It is be...,False,Who wrote the Qur'an in which Arabic language?
6323,Austin is the capital of the US state of Texas...,False,What is the largest man-made structure in the ...
6324,It became the 'Dravidakajagam' party under Nay...,False,Who was the first Chief Minister of Tamil Nadu?


In [122]:

# train_tr["contextMquestion"] = train_tr.apply(lambda x: x.columns, axis)

train_tr["contextMquestion"] = train_tr.apply(lambda x: list(set(map(str.lower,x["question_translated_tokenized"]))&set(map(str.lower,x["context_tokenized"]))), axis = 1)

In [123]:
train_tr[["contextMquestion", "answerable", "question_translated"]][train_tr["answerable"]== False]

,contextMquestion,answerable,question_translated
17,"[▁, ▁moon, ▁the]",False,Does the moon glow by itself?
36,"[▁rome, ▁in, ▁victori, ▁war, ▁the]",False,Was Rome victorious in the Phoenician War?
109,"[▁water, ▁on, ▁land, ▁and, ▁can]",False,Can crocodiles live on both land and water?
115,"[s, ▁m, ushroom, ▁can]",False,Can mushrooms heal wounds?
119,"[▁is, ▁university, ▁of, ▁oxford, ▁the]",False,Who is the founder of Oxford University?
...,...,...,...
6321,"[▁in, ▁, ▁was, ▁the]",False,In what country was the antidote to tuberculos...
6322,"[▁who, an, ▁in, ▁wrote, ', ▁, ▁the, ▁qur, arabic]",False,Who wrote the Qur'an in which Arabic language?
6323,"[▁is, -, ▁in, ▁of, ▁state, ▁, largest, ▁the, ▁...",False,What is the largest man-made structure in the ...
6324,"[▁minister, ▁who, ▁of, chief, ▁was, ▁, ▁the]",False,Who was the first Chief Minister of Tamil Nadu?


In [116]:
train_tr["question_translated_tokenized"]

0       [▁Who, ▁is, ▁the, ▁w, inner, ▁of, ▁the, ▁Th, i...
1                   [▁Who, ▁discovered, ▁X, -, ray, s, ?]
2       [▁When, ▁was, ▁the, ▁last, ▁Olympic, ▁Games, ▁...
3       [▁What, ', s, ▁the, ▁old, est, ▁broad, caster,...
4       [▁Where, ', s, ▁the, ▁Palest, inian, ▁capital, ?]
                              ...                        
6330    [▁When, ▁did, ▁Somalia, ▁make, ▁its, ▁second, ...
6331    [▁What, ▁was, ▁the, ▁world, ', s, ▁first, ▁tra...
6332    [▁Who, ▁is, ▁Egypt, ', s, ▁leader, ▁in, ▁2019, ?]
6333    [▁What, ▁is, ▁the, ▁most, ▁dens, ely, ▁pop, ul...
6334    [▁Does, ▁Reform, ed, ▁Bapt, ism, ▁have, ▁anyth...
Name: question_translated_tokenized, Length: 6335, dtype: object

In [ ]:
#!pip install -q collections

In [113]:
type(train_tr['context_tokenized'][0][0])

str

In [131]:
from collections import Counter

# Flatten both columns into one list of tokens
all_tokens = (
    train_tr['context_tokenized'].dropna().sum()
    + train_tr['question_translated_tokenized'].dropna().sum()
)

# Count token frequencies
token_counts = Counter(all_tokens)


[('▁', 117080), ('▁the', 48219), (',', 36479), ('▁of', 28205), ('.', 27649), ('s', 22796), ('▁and', 20146), ('▁in', 19208), ('a', 14249), ('▁to', 11641), ('▁is', 11447), ('ed', 9936), ('▁was', 8040), ('▁The', 7125), ('?', 6328), ('-', 6322), ('▁(', 6042), ('▁as', 5948), ("'", 5564), ('▁"', 5452), ('▁by', 5408), ('ly', 4649), ('e', 4152), ('ing', 4025), ('▁for', 4024), ('▁with', 3997), ('d', 3888), ('▁on', 3834), ('▁from', 3337), ('▁are', 3181), ('y', 3126), (')', 3093), ('"', 2954), ('▁that', 2919), ('▁first', 2553), ('▁an', 2539), ('which', 2281), ('▁or', 2274), ('▁at', 2263), ('▁be', 2167)]


In [142]:


# Get top N most frequent tokens
n = 50
top_tokens = token_counts.most_common(n)
top_tokens = list(set([word.lower() for word,_ in top_tokens]))

print(top_tokens)  # list of (token, count) tuples

['),', ')', 'ly', 'e', '▁on', 'y', '▁first', '▁(', 'ing', '▁at', '"', '▁', '▁to', '▁his', 'which', '▁be', "'", '▁"', '▁that', 'd', '.', '▁as', '▁from', '▁with', '▁is', '?', 'ed', '▁by', '▁has', '▁the', '▁and', ',', 'es', 'a', '▁for', '-', '▁in', '▁its', '▁of', 's', '▁are', '▁an', ';', '▁was', '▁it', '▁or', '▁what']


In [79]:
train_tr[["contextMquestion", "answerable", "question_translated"]][train_tr["answerable"]== False]

,contextMquestion,answerable,question_translated
17,[the],False,Does the moon glow by itself?
36,"[in, war, rome, the]",False,Was Rome victorious in the Phoenician War?
109,"[can, land, on, and, water]",False,Can crocodiles live on both land and water?
115,[can],False,Can mushrooms heal wounds?
119,"[of, university, is, the, oxford]",False,Who is the founder of Oxford University?
...,...,...,...
6321,"[in, was, the]",False,In what country was the antidote to tuberculos...
6322,"[wrote, who, the, in, qur'an, arabic]",False,Who wrote the Qur'an in which Arabic language?
6323,"[of, state, is, in, the, texas, largest]",False,What is the largest man-made structure in the ...
6324,"[was, of, who, the, chief, minister]",False,Who was the first Chief Minister of Tamil Nadu?


In [143]:
train_tr["conMquest_stripped"] = train_tr["contextMquestion"].apply(lambda wordlist: [word for word in wordlist if word not in top_tokens])

In [144]:
train_tr[["conMquest_stripped", "answerable", "question_translated"]][train_tr["answerable"]== False]


,conMquest_stripped,answerable,question_translated
17,[▁moon],False,Does the moon glow by itself?
36,"[▁rome, ▁victori, ▁war]",False,Was Rome victorious in the Phoenician War?
109,"[▁water, ▁land, ▁can]",False,Can crocodiles live on both land and water?
115,"[▁m, ushroom, ▁can]",False,Can mushrooms heal wounds?
119,"[▁university, ▁oxford]",False,Who is the founder of Oxford University?
...,...,...,...
6321,[],False,In what country was the antidote to tuberculos...
6322,"[▁who, an, ▁wrote, ▁qur, arabic]",False,Who wrote the Qur'an in which Arabic language?
6323,"[▁state, largest, ▁texas]",False,What is the largest man-made structure in the ...
6324,"[▁minister, ▁who, chief]",False,Who was the first Chief Minister of Tamil Nadu?


In [145]:
train_tr[["conMquest_stripped", "answerable", "question_translated"]][train_tr["answerable"]== True]

,conMquest_stripped,answerable,question_translated
0,[▁war],True,Who is the winner of the Thirty Years' War?
1,"[▁who, rays, ▁x, ▁discover]",True,Who discovered X-rays?
2,"[▁athens, olympic, ▁games, ▁held]",True,When was the last Olympic Games held in Athens?
3,"[est, ▁old, ▁world, broadcast, er]",True,What's the oldest broadcaster in the world?
4,[▁capital],True,Where's the Palestinian capital?
...,...,...,...
6329,"[▁transit, ▁seoul]",True,When did you start Seoul Transit?
6330,[tion],True,When did Somalia make its second inauguration?
6331,[],True,What was the world's first transportation system?
6332,"[▁who, ▁egypt]",True,Who is Egypt's leader in 2019?


In [75]:
train_tr["contextMquestion"].apply(lambda x: len(x))


0       3
1       3
2       6
3       5
4       2
       ..
6330    0
6331    3
6332    3
6333    8
6334    2
Name: contextMquestion, Length: 6335, dtype: int64

In [147]:
import pandas as pd


# Compute average lengths
avg_context_len = train_tr[train_tr["answerable"] == True]["conMquest_stripped"].apply(lambda x: len(x)).mean()
avg_question_len = train_tr[train_tr["answerable"] == False]["conMquest_stripped"].apply(lambda x: len(x)).mean()

print("Average contextMquestion length:", avg_context_len)
print("Average contextMquestion length False:", avg_question_len)


Average contextMquestion length: 2.5025117213663766
Average contextMquestion length False: 2.5234159779614327


In [40]:
#["question_stripped"])
train_tr["context_stripped"] = train_tr["context_stripped"]

0       ['The', 'conflict', 'between', 'France', 'and'...
1       ['X-rays', 'make', 'up', 'X-radiation', 'a', '...
2       ['In', '2022', 'Beijing', 'will', 'become', 't...
3       ['The', 'British', 'Broadcasting', 'Corporatio...
4       ['Palestine', '', '', 'officially', 'the', 'St...
                              ...                        
6330    ['In', 'February', '2012', 'Somali', 'governme...
6331    ['The', 'first', 'earth', 'tracks', 'were', 'c...
6332    ['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...
6333    ['Munich', '', '', '', 'is', 'the', 'capital',...
6334    ['Anabaptism', 'from', 'Neo-Latin', 'anabaptis...
Name: context_stripped, Length: 6335, dtype: object

In [41]:
train_tr[["contextMquestion", "answerable", "question_translated"]][train_tr["answerable"]== False]

,contextMquestion,answerable,question_translated
17,"list[{']', '[', ""'"", ' ', ','}]",False,Does the moon glow by itself?
36,"list[{']', '[', ""'"", ' ', ','}]",False,Was Rome victorious in the Phoenician War?
109,"list[{']', '[', ""'"", ' ', ','}]",False,Can crocodiles live on both land and water?
115,"list[{']', '[', ""'"", ' ', ','}]",False,Can mushrooms heal wounds?
119,"list[{']', '[', ""'"", ' ', ','}]",False,Who is the founder of Oxford University?
...,...,...,...
6321,"list[{']', '[', ""'"", ' ', ','}]",False,In what country was the antidote to tuberculos...
6322,"list[{']', '[', ""'"", ' ', ','}]",False,Who wrote the Qur'an in which Arabic language?
6323,"list[{']', '[', ""'"", ' ', ','}]",False,What is the largest man-made structure in the ...
6324,"list[{']', '[', ""'"", ' ', ','}]",False,Who was the first Chief Minister of Tamil Nadu?


In [95]:
from transformers import AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"

# Initialize the (fast) tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# --- Basics ---------------------------------------------------------------

text = "Hello, how are you?"
# enc = tokenizer(text)  # plain dict with 'input_ids' and 'attention_mask'
# print(enc.keys(), enc["input_ids"][:10])

# # With common options
enc = tokenizer(
    text,
    padding="max_length",  # or True / "longest"
    truncation=True,
    max_length=64,
    return_tensors="pt"    # "pt" | "tf" | "np"
)
print(enc.input_ids.shape, enc.attention_mask.shape)
#
# # Batch encode
# batch = tokenizer(
#     ["Hello!", "This is a longer sentence."],
#     padding=True,
#     truncation=True,
#     return_tensors="pt"
# )
#
# # Decode ids back to text
decoded = tokenizer.decode(enc["input_ids"][0], skip_special_tokens=True)
print("decoded", decoded)

ModuleNotFoundError: No module named 'spacy'

In [148]:

train_tr.to_csv("data/train_translated_tokenized.csv")

In [102]:
train_tr

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated,question_translated_stripped,question_translated_wordcount,contextMquestion,conMquest_stripped,question_translated_tokenized,context_tokenized
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"[30년, 전쟁의, 승자는, 누구인가]",4,"[The, conflict, between, France, and, Spain, c...",108,Who is the winner of the Thirty Years' War?,"[Who, is, the, winner, of, the, Thirty, Years,...",9,"[war, of, the]",[war],"[▁Who, ▁is, ▁the, ▁w, inner, ▁of, ▁the, ▁Th, i...","[▁The, ▁conflict, ▁between, ▁France, ▁and, ▁Sp..."
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"[엑스선은, 누가, 발견하였는가]",3,"[X-rays, make, up, X-radiation, a, form, of, e...",122,Who discovered X-rays?,"[Who, discovered, X-rays]",3,"[discovered, who, x-rays]","[discovered, x-rays]","[▁Who, ▁discovered, ▁X, -, ray, s, ?]","[▁X, -, ray, s, ▁make, ▁up, ▁X, -, radi, ation..."
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"[아테네에서, 언제, 가장, 최근의, 올림픽이, 올렸나요]",6,"[In, 2022, Beijing, will, become, the, first-e...",197,When was the last Olympic Games held in Athens?,"[When, was, the, last, Olympic, Games, held, i...",9,"[athens, in, the, olympic, held, games]","[athens, olympic, held, games]","[▁When, ▁was, ▁the, ▁last, ▁Olympic, ▁Games, ▁...","[▁In, ▁202, 2,, ▁Beijing, ▁will, ▁become, ▁the..."
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"[세상에서, 가장, 오래된, 방송사는, 무엇인가]",5,"[The, British, Broadcasting, Corporation, BBC,...",70,What's the oldest broadcaster in the world?,"[What's, the, oldest, broadcaster, in, the, wo...",7,"[broadcaster, the, world, in, oldest]","[broadcaster, world, oldest]","[▁What, ', s, ▁the, ▁old, est, ▁broad, caster,...","[▁The, ▁British, ▁Broad, casting, ▁Corporation..."
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"[팔레스타인, 수도는, 어딘가요]",3,"[Palestine, , , officially, the, State, of, Pa...",84,Where's the Palestinian capital?,"[Where's, the, Palestinian, capital]",4,"[capital, the]",[capital],"[▁Where, ', s, ▁the, ▁Palest, inian, ▁capital, ?]","[▁Palestine, ▁(, ▁', ),, ▁offici, ally, ▁the, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"[소말리아는, 2차, 개헌을, 언제, 했나요]",5,"[In, February, 2012, Somali, government, offic...",181,When did Somalia make its second inauguration?,"[When, did, Somalia, make, its, second, inaugu...",7,[],[],"[▁When, ▁did, ▁Somalia, ▁make, ▁its, ▁second, ...","[▁In, ▁February, ▁2012,, ▁Somali, ▁government,..."
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"[세상에서, 가장, 먼저, 시작된, 교통수단은, 무엇인가]",6,"[The, first, earth, tracks, were, created, by,...",118,What was the world's first transportation system?,"[What, was, the, world's, first, transportatio...",7,"[first, was, the]",[],"[▁What, ▁was, ▁the, ▁world, ', s, ▁first, ▁tra...","[▁The, ▁first, ▁earth, ▁tra, cks, ▁were, ▁crea..."
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"[2019년, 이집트의, 지도자는, 누구인가]",4,"[Abdel, Fattah, Saeed, Hussein, Khalil, El-Sis...",30,Who is Egypt's leader in 2019?,"[Who, is, Egypt's, leader, in, 2019]",6,"[is, who, in]",[],"[▁Who, ▁is, ▁Egypt, ', s, ▁leader, ▁in, ▁2019, ?]","[▁Ab, del, ▁Fat, tah, ▁Sae, ed, ▁Hussein, ▁Kha..."
6333,15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,NaN,"[독일에서, 가장, 인구밀도가, 높은, 도시는, 무엇인가]",6,"[Munich, , , , is, the, capital, and, most, 

In [ ]:
import spacy
nlp = spacy.load('en')

def vbd(token):
   """a bad conjugation function"""
   if token.pos_ == 'VERB':
       return token.lemma_ + 'ed'
spacy.tokens.Token.set_extension('vbd', getter=vbd, default=None)
doc = nlp(u'Apple is looking at buying U.K. startup for $1 billion')
for token in doc:
     print(token.text, ":", token._.vbd)

In [86]:
pred no
when -> numbers or month names
where -> big letter, not a name -> maybe find a dict of places
Who -> names/nouns
what -> noun/an object/also can be a location
how -> adjectives ------ how many -> a number


SyntaxError: invalid syntax (2435472204.py, line 1)

In [37]:
def count_translated_quest_tokens(df, topn: int = 5): #irrelevant
    langs = ["ko", "ar", "te"]
    col  = "question_translated"
    for l in langs:
        mask = df["lang"] == l
        lang_df = df.loc[mask].copy()
        if lang_df[col].dtype == object and lang_df[col].map(lambda x: isinstance(x, list)).any():
            tokens = list(chain.from_iterable(x or [] for x in lang_df[col].tolist()))
        else:
            tokens = list(chain.from_iterable((str(x or "")).split() for x in lang_df[col].tolist()))
        tokens = [t.strip() for t in tokens if isinstance(t, str) and t.strip()]
        counts = pd.Series(tokens).value_counts().head(topn)

        out = counts.reset_index()
        out.columns = ["token", "count"]
        return out #


In [39]:
train_tr[train_tr["answerable"] == True]

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6329,15337,서울교통공사는 언제 창단했나요?,Seoul Metro (Hangul: 서울메트로) is a public corpor...,ko,True,111,1970,NaN,"['서울교통공사는', '언제', '창단했나요']",3,"['Seoul', 'Metro', 'Hangul', '서울메트로', 'is', 'a...",47,When did you start Seoul Transit?
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"['소말리아는', '2차', '개헌을', '언제', '했나요']",5,"['In', 'February', '2012', 'Somali', 'governme...",181,When did Somalia make its second inauguration?
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"['세상에서', '가장', '먼저', '시작된', '교통수단은', '무엇인가']",6,"['The', 'first', 'earth', 'tracks', 'were', 'c...",118,What was the world's first transportation system?
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"['2019년', '이집트의', '지도자는', '누구인가']",4,"['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...",30,Who is Egypt's leader in 2019?


In [40]:
train_tr[train_tr["answerable"] == False]

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
17,4809,달은 자체발광하는가?,Moonlight consists of mostly sunlight (with li...,ko,False,-1,no,NaN,"['달은', '자체발광하는가']",2,"['Moonlight', 'consists', 'of', 'mostly', 'sun...",21,Does the moon glow by itself?
36,4828,포에니 전쟁에서 로마가 승리했나요?,With the end of the Macedonian Wars – which ra...,ko,False,-1,no,NaN,"['포에니', '전쟁에서', '로마가', '승리했나요']",4,"['With', 'the', 'end', 'of', 'the', 'Macedonia...",78,Was Rome victorious in the Phoenician War?
109,4901,악어는 땅과 물 둘 다에서 살 수 있을까?,"When on land, an American alligator moves eith...",ko,False,-1,no,NaN,"['악어는', '땅과', '물', '둘', '다에서', '살', '수', '있을까']",8,"['When', 'on', 'land', 'an', 'American', 'alli...",147,Can crocodiles live on both land and water?
115,4907,곰팡이가 상처를 치료할 수 있을까?,Many species produce metabolites that are majo...,ko,False,-1,no,NaN,"['곰팡이가', '상처를', '치료할', '수', '있을까']",5,"['Many', 'species', 'produce', 'metabolites', ...",301,Can mushrooms heal wounds?
119,4911,옥스퍼드 대학의 창립주는 누구인가요?,The University of Oxford has no known foundati...,ko,False,-1,no,NaN,"['옥스퍼드', '대학의', '창립주는', '누구인가요']",4,"['The', 'University', 'of', 'Oxford', 'has', '...",109,Who is the founder of Oxford University?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6321,15322,క్షయ వ్యాధికి విరుగుడు ఏ దేశంలో కనుగొన్నారు?,Vaccines against anthrax for use in livestock ...,te,False,-1,France,ఫ్రాన్స్,"['క్షయ', 'వ్యాధికి', 'విరుగుడు', 'ఏ', 'దేశంలో'...",6,"['Vaccines', 'against', 'anthrax', 'for', 'use...",100,In what country was the antidote to tuberculos...
6322,15323,ఖురాన్ ఏ అరబ్బీ భాషలో ఎవరు రాసారు?,are broken Other Names of the Qur'an: It is be...,te,False,-1,Prophet Muhammad,ముహమ్మద్ ప్రవక్త,"['ఖురాన్', 'ఏ', 'అరబ్బీ', 'భాషలో', 'ఎవరు', 'రా...",6,"['are', 'broken', 'Other', 'Names', 'of', 'the...",139,Who wrote the Qur'an in which Arabic language?
6323,15324,టెక్సస్ రాష్ట్రంలోని అతిపెద్ద మానవ నిర్మితం ఏది ?,Austin is the capital of the US state of Texas...,te,False,-1,JP Morgan Chase Tower,జేపీ మోర్గాన్ ఛేజ్ టవర్,"['టెక్సస్', 'రాష్ట్రంలోని', 'అతిపెద్ద', 'మానవ'...",7,"['Austin', 'is', 'the', 'capital', 'of', 'the'...",135,What is the largest man-made structure in the ...
6324,15325,తమిళనాడులో రాష్ట్ర మొదటి ముఖ్యమంత్రి ఎవరు?,It became the 'Dravidakajagam' party under Nay...,te,False,-1,C. N. Annadurai,సి.ఎన్.అన్నాదురై,"['తమిళనాడులో', 'రాష్ట్ర', 'మొదటి', 'ముఖ్యమంత్ర...",5,"['It', 'became', 'the', 'Dravidakajagam', 'par...",143,Who was the first Chief Minister of Tamil Nadu?


### Idea: look at cosine difference between question and answer, but it will very likely fail. Other idea is too look at how questions are formed and if it is possible to extract certain information for the context. My theory that certain types of starting tockens correspond to non-answerable questions.

In [ ]:
train_set

In [ ]:
train_ds = load_dataset(
    "parquet",
    data_files={"train": "data/train.parquet"}
)["train"]

val_ds = load_dataset(
    "parquet",
    data_files={"validation": "data/validation.parquet"}
)["validation"]

test_ds = load_dataset(
    "json",
    data_files={"test": "data/test.json"}
)["test"]

In [ ]:
train_ds